
# Lecture 1 (2 hours): ML Paradigms, Labels, Loss, Overfitting & Regularisation (Python Lab)

**Audience:** 2nd-year BSc Data Science  
**Goal:** Learn core ML paradigms and *experience* loss functions, overfitting, and regularisation hands-on.

---

## Contents
1. Setup & imports  
2. ML paradigms + what are labels? (quick data inspection)  
3. Loss functions (MSE from scratch)  
4. Overfitting vs underfitting (polynomial regression demo)  
5. Regularisation (Ridge & Lasso)  
6. Reflection questions

> Tip: Run cells in order. Most exercises have a **TODO** section for students.



## 1) Setup

We'll use:
- `numpy`, `pandas`, `matplotlib`
- `scikit-learn` for datasets and baseline models

Run the cell below to import everything.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, log_loss, accuracy_score



## 2) ML paradigms + labels (supervised vs unsupervised vs reinforcement learning)

### Quick conceptual recap
- **Supervised learning:** data comes with labels \(y\). Learn a mapping \(X \to y\).  
  - Regression: \(y\) continuous (e.g., price)
  - Classification: \(y\) categorical (e.g., spam/ham)
- **Unsupervised learning:** no labels. Learn structure in \(X\) (e.g., clusters).
- **Reinforcement learning:** learn actions via rewards from interaction (agent–environment–reward).

### In practice
A typical *supervised* dataset has:
- **Features**: a matrix \(X\) of shape `(n_samples, n_features)`
- **Labels**: a vector \(y\) of shape `(n_samples,)`

We'll start with the classic **diabetes** dataset (regression).


In [ ]:
diabetes = load_diabetes(as_frame=True)
X = diabetes.data
y = diabetes.target

print("X shape:", X.shape)
print("y shape:", y.shape)
display(X.head())
display(y.head())



### Exercise A (labels): turn regression into binary classification

Create a binary label:
- `1` if the target is above the median
- `0` otherwise

Then check class balance.


In [ ]:
# TODO: create a binary label y_bin using the median of y
median_y = np.median(y)
y_bin = (y > median_y).astype(int)

print("Median of y:", median_y)
print("Class counts:", np.bincount(y_bin))
print("Proportion of class 1:", y_bin.mean())



## 3) Loss functions: what models optimise

A **loss function** measures how bad predictions are *per sample*.
An **objective** usually aggregates the loss over a dataset (e.g., mean loss).

For regression, common losses:
- **Squared error:** \((y - \hat{y})^2\)
- **Absolute error:** \(|y - \hat{y}|\)

We'll implement **Mean Squared Error (MSE)** from scratch and compare with scikit-learn.


In [ ]:
def mse_from_scratch(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean((y_true - y_pred)**2)

# Simple baseline predictor: always predict the mean of y
y_pred_baseline = np.full_like(y, fill_value=np.mean(y), dtype=float)

print("MSE (scratch):", mse_from_scratch(y, y_pred_baseline))
print("MSE (sklearn):", mean_squared_error(y, y_pred_baseline))



### Visual demo: MSE as a function of a single parameter

We fit the simplest possible model: predict a constant \(c\) for everyone.  
The MSE depends on \(c\). We'll plot MSE over a range of \(c\) values and verify the minimum is at the mean of \(y\).


In [ ]:
c_values = np.linspace(y.min(), y.max(), 200)
mse_values = [mse_from_scratch(y, np.full_like(y, c, dtype=float)) for c in c_values]

plt.figure()
plt.plot(c_values, mse_values)
plt.axvline(np.mean(y), linestyle="--")
plt.title("MSE vs constant prediction c")
plt.xlabel("c")
plt.ylabel("MSE")
plt.show()

print("Mean(y) =", np.mean(y))
print("Argmin approx c =", c_values[int(np.argmin(mse_values))])



## 4) Overfitting vs underfitting (polynomial regression)

We'll build a 1D synthetic dataset so the effect is visible.

**Idea:**  
- Low-degree polynomial → too simple → **underfitting**
- Very high-degree polynomial → too flexible → **overfitting**

We'll:
1. Generate data
2. Split into train/test
3. Fit polynomial regression models of different degrees
4. Compare train vs test error


In [ ]:
rng = np.random.default_rng(42)

n = 60
X_1d = np.linspace(-3, 3, n)
y_true_fn = np.sin(X_1d)  # underlying function
noise = rng.normal(0, 0.2, size=n)
y_1d = y_true_fn + noise

X_1d = X_1d.reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X_1d, y_1d, test_size=0.3, random_state=42
)

plt.figure()
plt.scatter(X_train, y_train, label="train")
plt.scatter(X_test, y_test, label="test")
plt.title("Synthetic dataset (train/test split)")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()



### Fit polynomial regression of varying degrees

We use a pipeline:
- PolynomialFeatures(degree=d)
- LinearRegression()

We'll compute:
- Train MSE
- Test MSE

and plot both vs degree.


In [ ]:
degrees = list(range(1, 21))
train_mse = []
test_mse = []

for d in degrees:
    model = Pipeline([
        ("poly", PolynomialFeatures(degree=d, include_bias=False)),
        ("linreg", LinearRegression())
    ])
    model.fit(X_train, y_train)
    
    yhat_train = model.predict(X_train)
    yhat_test = model.predict(X_test)
    
    train_mse.append(mean_squared_error(y_train, yhat_train))
    test_mse.append(mean_squared_error(y_test, yhat_test))

plt.figure()
plt.plot(degrees, train_mse, label="Train MSE")
plt.plot(degrees, test_mse, label="Test MSE")
plt.title("Overfitting demo: Train vs Test MSE by polynomial degree")
plt.xlabel("Polynomial degree")
plt.ylabel("MSE")
plt.legend()
plt.show()

best_degree = degrees[int(np.argmin(test_mse))]
print("Best test MSE at degree:", best_degree)



### Visualise a few degrees

Plot fitted curves for degrees that typically show:
- Underfit (e.g. degree 1)
- Reasonable (e.g. around the best degree)
- Overfit (e.g. degree 15 or 20)

Feel free to change the degrees below.


In [ ]:
plot_degrees = [1, best_degree, 15]

x_grid = np.linspace(-3, 3, 400).reshape(-1, 1)

plt.figure()
plt.scatter(X_train, y_train, label="train")
plt.scatter(X_test, y_test, label="test")

for d in plot_degrees:
    model = Pipeline([
        ("poly", PolynomialFeatures(degree=d, include_bias=False)),
        ("linreg", LinearRegression())
    ])
    model.fit(X_train, y_train)
    plt.plot(x_grid, model.predict(x_grid), label=f"degree {d}")

plt.title("Polynomial regression fits")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()



## 5) Regularisation (Ridge & Lasso)

**Regularisation** adds a penalty that discourages overly complex models.

- **Ridge (L2):** penalises large coefficients, usually keeps all features
- **Lasso (L1):** can drive some coefficients to exactly zero (feature selection behaviour)

We'll revisit the polynomial regression case and compare:
- Unregularised linear regression on polynomial features
- Ridge
- Lasso

Important: regularisation strength depends on feature scale, so we standardise.


In [ ]:
def fit_and_score(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    return (
        mean_squared_error(y_train, model.predict(X_train)),
        mean_squared_error(y_test, model.predict(X_test)),
    )

degree = 15  # intentionally high to invite overfitting
alphas = [0.001, 0.01, 0.1, 1.0, 10.0]

# Unregularised baseline
baseline = Pipeline([
    ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
    ("scaler", StandardScaler()),
    ("reg", LinearRegression())
])

base_train_mse, base_test_mse = fit_and_score(baseline, X_train, y_train, X_test, y_test)
print(f"Baseline (no regularisation), degree={degree}: train MSE={base_train_mse:.4f}, test MSE={base_test_mse:.4f}")

ridge_results = []
lasso_results = []

for a in alphas:
    ridge = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("scaler", StandardScaler()),
        ("reg", Ridge(alpha=a))
    ])
    lasso = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("scaler", StandardScaler()),
        ("reg", Lasso(alpha=a, max_iter=20000))
    ])
    ridge_results.append((a, ) + fit_and_score(ridge, X_train, y_train, X_test, y_test))
    lasso_results.append((a, ) + fit_and_score(lasso, X_train, y_train, X_test, y_test))

ridge_df = pd.DataFrame(ridge_results, columns=["alpha", "train_MSE", "test_MSE"])
lasso_df = pd.DataFrame(lasso_results, columns=["alpha", "train_MSE", "test_MSE"])

display(ridge_df)
display(lasso_df)



### Plot test MSE vs regularisation strength

We plot MSE as a function of `alpha` (regularisation strength).  
Note: alpha is on a log scale in many ML contexts.


In [ ]:
plt.figure()
plt.plot(ridge_df["alpha"], ridge_df["test_MSE"], marker="o", label="Ridge test MSE")
plt.plot(lasso_df["alpha"], lasso_df["test_MSE"], marker="o", label="Lasso test MSE")
plt.xscale("log")
plt.title("Effect of regularisation strength (degree=15)")
plt.xlabel("alpha (log scale)")
plt.ylabel("Test MSE")
plt.legend()
plt.show()



## 6) Reflection / check-your-understanding questions

1. In supervised learning, what changes when you change the definition of the label \(y\)?  
2. Why is MSE minimised at the mean when predicting a constant?  
3. In the polynomial regression demo, why does training MSE typically decrease as degree increases?  
4. Why can test MSE start increasing as degree increases?  
5. What does regularisation do to the fitted coefficients, and why can that improve test performance?  
6. Compare Ridge vs Lasso qualitatively—when might you prefer one over the other?
